In [1]:
import os, shutil

def find(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

src = find("11_train_student_lat.py")
shutil.copy(src, "/kaggle/working/11_train_student_lat.py")
print("Script:", src)
for f in ["_meta_crop280.csv", "knee_side.csv", "oof_effb0_kf2.csv", "report_labels.csv"]:
    print(f"{f:20s} -> {find(f)}")

Script: /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/11_train_student_lat.py
_meta_crop280.csv    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-31-preprocess-crop280/vol_train_crop280/_meta_crop280.csv
knee_side.csv        -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-22-sisi-lutut/knee_side.csv
oof_effb0_kf2.csv    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-06-tahapc-teacher-oof/oof_effb0_kf2.csv
report_labels.csv    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --oof oof_effb0_kf2.csv --side_csv knee_side.csv \
        --backbone vit_small_patch14_dinov2.lvd142m --lr_backbone 1e-4 --llrd 0.8 --epochs 12 \
        --meta _meta_crop280.csv --img_size 280 --batch 2 --accum 2"

# GPU 0: seed 42
CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $COMMON --seed 42 \
    --tag dinov2s_student_v1d_lat_crop280 > log_dinov2s_student_v1d_lat_crop280.txt 2>&1 &

# GPU 1: seed 7
CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $COMMON --seed 7 \
    --tag dinov2s_student_v1d_lat_crop280_s7 > log_dinov2s_student_v1d_lat_crop280_s7.txt 2>&1 &
wait
for t in dinov2s_student_v1d_lat_crop280 dinov2s_student_v1d_lat_crop280_s7; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Volume\|Normalisasi\|Train \|ukuran\|Error\|error" log_*.txt

===== dinov2s_student_v1d_lat_crop280 =====
Epoch 11: loss 0.3709 | val AUC 0.8869 | val-ref AUC 0.8996 | gold AUC 0.8651 | 22.0 menit

=== [dinov2s_student_v1d_lat_crop280] Checkpoint terbaik (val-ref AUC 0.9000) pada label emas ===
ACL                AUC=0.925
MCL                AUC=0.896
Medial Meniscus    AUC=0.887
Lateral Meniscus   AUC=0.784
Medial OA          AUC=0.974
Lateral OA         AUC=0.812
PF OA              AUC=0.803
Effusion           AUC=0.868
Synovitis          AUC=0.724
Baker's            AUC=0.955
Contusion          AUC=0.903
Fracture           AUC=0.883
Mean AUC gold: 0.8678
===== dinov2s_student_v1d_lat_crop280_s7 =====
Epoch 11: loss 0.3703 | val AUC 0.8824 | val-ref AUC 0.8987 | gold AUC 0.8677 | 26.2 menit

=== [dinov2s_student_v1d_lat_crop280_s7] Checkpoint terbaik (val-ref AUC 0.8988) pada label emas ===
ACL                AUC=0.911
MCL                AUC=0.892
Medial Meniscus    AUC=0.853
Lateral Meniscus   AUC=0.775
Medial OA          AUC=0.981
Lateral OA 

In [3]:
import pandas as pd
for tag in ["dinov2s_student_v1d_lat_crop280", "dinov2s_student_v1d_lat_crop280_s7"]:
    print(f"\n===== {tag} =====")
    try:
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== dinov2s_student_v1d_lat_crop280 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5393   0.7754       0.7858    0.8098 21.9689
     1 0.4650   0.8308       0.8454    0.8349 21.9639
     2 0.4418   0.8461       0.8631    0.8439 21.9942
     3 0.4313   0.8614       0.8774    0.8574 21.9989
     4 0.4198   0.8613       0.8768    0.8505 22.0121
     5 0.4108   0.8721       0.8872    0.8545 21.9851
     6 0.4015   0.8790       0.8929    0.8612 21.9832
     7 0.3925   0.8819       0.8969    0.8599 21.9722
     8 0.3848   0.8834       0.8972    0.8624 21.9918
     9 0.3774   0.8871       0.9000    0.8678 22.0254
    10 0.3729   0.8867       0.8995    0.8631 22.0439
    11 0.3709   0.8869       0.8996    0.8651 22.0342

===== dinov2s_student_v1d_lat_crop280_s7 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5299   0.7617       0.7745    0.8061 26.2914
     1 0.4644   0.8183       0.8347    0.8412 26.2174
     2 0.4432   0.8415       0.8583    0.85